# Wektorowa baza danych na przykładzie Qdrant

W celu zapisu embeddingów, należy użyć bazy danych, która wspiera przechowywanie wektorów. 

Przykładem takiej bazy danych jest Qdrant.

Wykonaj poniższą komendę w katalogu root projektu:
```
docker compose up -d
```

http://localhost:6333/dashboard#/collections

## Połączenie z bazą

In [1]:
import numpy as np
from qdrant_client import models, QdrantClient
from qdrant_client.models import VectorParams, Distance

/home/patryk/CloudDrive/Szkolenia/Realizacje/Coderslab/Samsung_grupa_2/samsung-grupa-2-rag/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
client = QdrantClient(url="http://localhost:6333")
client

/home/patryk/CloudDrive/Szkolenia/Realizacje/Coderslab/Samsung_grupa_2/samsung-grupa-2-rag/.venv/lib/python3.14/site-packages/qdrant_client/qdrant_remote.py:284: UserWarning: Qdrant client version 1.19.1 is incompatible with server version 1.16.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


## Tworzenie kolekcji

In [3]:
client.create_collection("test_collection",
                         vectors_config=VectorParams(size=5, distance=Distance.COSINE))

True

## Dodawanie i modyfikacja rekordów

Aby dodać lub zmodyfikować rekordy w kolekcji używamy funkcji `upsert`.

Jeśli miarą odległości, której używamy będzie `cosine`, to dodane wektory ulegną normalizacji – długość wektora będzie jednostkowa, przy zachowaniu proporcji między składowymi.

**Dodawanie rekordów**

In [6]:
points = [
    models.PointStruct(
        id=1, payload={"a": [1, 2, 3], "b": 30}, vector=[1, 2, 3, 4, 5]
    ),
    
    models.PointStruct(
        id=2, payload={"a": [10, 20, 30], "b": 40}, vector=np.array([10, 20, 30, 40, 50])
    ),
]

points

[PointStruct(id=1, vector=[1.0, 2.0, 3.0, 4.0, 5.0], payload={'a': [1, 2, 3], 'b': 30}),
 PointStruct(id=2, vector=[10.0, 20.0, 30.0, 40.0, 50.0], payload={'a': [10, 20, 30], 'b': 40})]

In [7]:
client.upsert("test_collection", points=points)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

In [ ]:
[0.13483997,0.26967993,0.40451992,0.53935987,0.6741998]

**Modyfikacja rekordów**

Aby zmodyfikować rekordy, należy dodać rekordy o takim samym id.

W poniższym przykładzie rekord o id=2 zostanie zmodyfikowany. Przy okazji dodamy rekord o id=3.

In [ ]:
points = [
    models.PointStruct(
        id=2, payload={"a": [10, 20, 30], "b": 50}, vector=np.array([-10, -20, -30, -40, -50])
    ),
    
    models.PointStruct(
        id=3, payload={"a": {"list": [10, 20, 30], "string": "hello"}}, vector=[-101, -201, -301, -401, -501]
    ),
]

points

In [ ]:
client.upsert("test_collection", points=points)

## Wyszukiwanie podobnych wektorów

In [ ]:
search_results = client.query_points(
    collection_name="test_collection",
    query=[0.1, 0.2, 0.3, 0.4, 0.5],
    with_vectors=True,
    with_payload=False,
    limit=2
)

In [ ]:
search_results.points

In [ ]:
for vector in search_results.points:
    print(vector.vector)

## Usuwanie rekordów oraz kolekcji

### Usunięcie rekordu

In [8]:
client.delete(
    collection_name="test_collection",
    points_selector=models.PointIdsList(points=[1]),
)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

### Usunięcie kolekcji

In [9]:
client.delete_collection("test_collection")

True

> **ZADANIA**

## Qdrant w LangChain

Aby zapisać do bazy Qdrant zembedowane chunki należy użyć klasy `QdrantVectorStore`.

Najpierw należy utworzyć kolekcję niezależnie od LangChaina a następnie utworzyć obiekt vector_store.

In [10]:
import os
from dotenv import load_dotenv
from langchain_qdrant import QdrantVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()

True

In [11]:
model_name = "sdadas/mmlw-roberta-base"

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={'device': 'cpu'},
    query_encode_kwargs={"prompt": "zapytanie: "}
)

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 5198.05it/s]


In [14]:
client.create_collection("knowledge_database",
                         vectors_config=VectorParams(size=768, distance=Distance.COSINE))

True

In [15]:
vector_store = QdrantVectorStore(
    client=client,
    collection_name="knowledge_database",
    embedding=embeddings,
)

**`add_texts`** – przyjmuje listę tekstów i osobną listę metadanych.

In [16]:
texts = [
    "Andrzej ma psa.",
    "Bartek ma kota.",
    "Czarek ma królika.",

    "Andrzej jeździ samochodem.",
    "Bartek jeździ rowerem.",
    "Czarek jeździ tramwajem.",
]

metadatas = [
    {"length": len(text), "person": text.split()[0]} for text in texts
]

In [17]:
metadatas

[{'length': 15, 'person': 'Andrzej'},
 {'length': 15, 'person': 'Bartek'},
 {'length': 18, 'person': 'Czarek'},
 {'length': 26, 'person': 'Andrzej'},
 {'length': 22, 'person': 'Bartek'},
 {'length': 24, 'person': 'Czarek'}]

In [18]:
vector_store.add_texts(texts, metadatas)

['0abd35abd3fc4cf89d3d3f4d1e34772a',
 'd943ac639b81481cbf6f85f53ed8e2d2',
 '35023a48f55140879670264b639b7533',
 '3cac246ea05e45ec9b65626a28d71635',
 '7b82523487e74cffb46775363bd4d85c',
 'd74498e69bfd4d86a5de1521372b4c7a']

**`add_documents`** – przyjmuje listę obiektów `Document`, w których tekst i metadane są razem. Tak wyglądają np. chunki zwracane przez `splitter.split_documents()`, więc można je dodać do bazy bezpośrednio.

In [20]:
from langchain_core.documents import Document

documents = [
    Document(page_content=text, metadata={"length": len(text), "person": text.split()[0]})
    for text in [
        "Andrzej lubi pływać.",
        "Bartek lubi biegać.",
        "Czarek lubi grać w szachy.",
    ]
]

documents

[Document(metadata={'length': 20, 'person': 'Andrzej'}, page_content='Andrzej lubi pływać.'),
 Document(metadata={'length': 19, 'person': 'Bartek'}, page_content='Bartek lubi biegać.'),
 Document(metadata={'length': 26, 'person': 'Czarek'}, page_content='Czarek lubi grać w szachy.')]

In [21]:
vector_store.add_documents(documents)

['40a4ee30f73a4632bbe044d262d22fbe',
 '251ee4aaa210403992afc566875dbedb',
 '746d937e5f1b4301a63cd0bc1ac26ba4']

**Uwaga:** Zaletą tego podejścia jest to, że szybko można zembeddować chunki i dodać je do bazy. Ale jeśli z jakiegoś powodu zależy nam na rozdzieleniu tych dwóch etapów, to lepiej użyć innego sposobu.

> **ZADANIA**